<a href="https://colab.research.google.com/github/LSim2002/AI-DDQN-stock-trading/blob/main/pfe_v2_8_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
import csv
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split
import itertools
import os
import json

import random
import copy

import gc

import numpy as np

from tqdm.auto import tqdm

from prettytable import PrettyTable

In [ ]:
!pip install captum
from captum.attr import Saliency, IntegratedGradients, InputXGradient


# Define SimpleCNN model

In [ ]:
# Define a simplified CNN model
class SimpleCNN(nn.Module):
    def __init__(self, n_classes,sizex,sizey):
        super(SimpleCNN, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(32 * int(sizex/4) * int(sizey/4), 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.model(x)

# Functions

## Check RAM Usage Function(restart colab session to free ram)

In [ ]:
def check_ram():
  # Collect all tensors stored in GPU memory
  gpu_tensors = [obj for obj in gc.get_objects() if isinstance(obj, torch.Tensor) and obj.device.type == "cuda"]

  # Sort tensors by memory size (descending order)
  gpu_tensors_sorted = sorted(gpu_tensors, key=lambda t: t.numel() * t.element_size(), reverse=True)

  # Print details of each tensor
  print(f"{'Index':<5} {'Shape':<25} {'Dtype':<10} {'Size (MB)':<10}")
  print("=" * 60)
  for i, tensor in enumerate(gpu_tensors_sorted):
      size_mb = tensor.numel() * tensor.element_size() / 1024**2
      print(f"{i:<5} {str(tensor.shape):<25} {str(tensor.dtype):<10} {size_mb:<10.2f}")

#check_ram()

## Generate images function


Instead of storing the actual images, each entry should be:

images_md[i] = [noise_seed, zone_0_color, zone_1_color, ...]

Where:

  noise_seed: A random seed (integer) to regenerate the same noise each time.

  zone_X_color: the color for the red channel which is the only one that depends on the class (if truthful). The randomized G and B values for the zones are obtained and reproductible via the noise_seed.

In [ ]:
# combined_images[i] = [noise_seed, zone_0_red, zone_1_red, ...] #the noise is defined by nosie seed, the g and b channls for the zones are randomized and defined by the noise seed as well
def generate_images_md(n_images, sizex, sizey, n_noise_pixels, zone_truthfulness_list,
                        zone_range_list, n_classes, device, seed): #the seed is for when we generate the random labels and when we choose if a zone is truthful or randomized
    """
    Returns:
        torch.Tensor: Metadata for image generation, of shape [n_images, 1+num_zones]. Each image in the tensor is like this [noise_seed, zone_0_red, zone_1_red, ...].
        torch.Tensor: Corresponding labels tensor of shape (n_images,).
    """

    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)

    # Init ready labels
    labels = torch.randint(0, n_classes, (n_images,), dtype=torch.uint8, device=device)

    # Init empty metadata_tensor
    metadata_tensor = torch.zeros((n_images, 1 + len(zone_truthfulness_list)), dtype=torch.uint8, device=device) # each image_md is [noise_seed, red_channel_zone_0, red_channel_zone1 etc]

    # Fill noise_seeds in metadata_tensor
    metadata_tensor[:, 0] = torch.arange(n_images, dtype=torch.uint8, device=device)

    # Precompute class-dependent red channel values
    red_values_list = [torch.linspace(range[0], range[1], n_classes, dtype=torch.uint8, device=device)
                       for range in zone_range_list] #this contains the red values for each class, for each zone (different zone have different differenciation steps)

    # Generate red channel values for zones efficiently by iterating over each zone
    for zone_id, truthfulness in tqdm(enumerate(zone_truthfulness_list), desc="Generating zone metadata"):
        mask = torch.rand(n_images, device=device) < truthfulness
        correct_red_values = red_values_list[zone_id].gather(0, labels.long())
        random_red_values = torch.randint(0, 256, (n_images,), dtype=torch.uint8, device=device)
        metadata_tensor[:, zone_id + 1] = torch.where(mask, correct_red_values, random_red_values)



    print(f"Generated metadata for {n_images} images with {len(zone_truthfulness_list)} zones each.")
    return metadata_tensor, labels

## Create DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch.utils.data import random_split

class md_imageDataset(Dataset): #dataset is not already normalized, uint8 type
    def __init__(self, images, labels):
        """
        Args:
            data (np.ndarray): ND RGB images of shape [num_images, 1 + num_zones, 224, 224, 3].
        """
        self.images_md = images
        self.labels = labels

        assert self.images_md.shape[0] == self.labels.shape[0], "Data and labels must have the same length"

    def __len__(self):
        return self.images_md.shape[0]

    def __getitem__(self, idx):
        return self.images_md[idx], self.labels[idx] # sape of images md: (n_images, ,)




def create_data_loaders(dataset, train_size, val_size, test_size, batch_size, seed=None):

    assert train_size + val_size + test_size == 1, "Train, validation, and test sizes must sum to 1"

    # Create a generator for reproducibility
    generator = torch.Generator()
    if seed is not None:
        generator.manual_seed(seed)

    # Calculate dataset splits
    total_size = len(dataset)
    train_size = int(total_size * train_size)
    val_size = int(total_size * val_size)
    test_size = total_size - train_size - val_size

    # Split the dataset
    train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size], generator=generator)

    # Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

## Process image function

Because we're storing as little data as possible for each image, including the random seed, we need to iterate over each image (can't paralelize using different random seeds for each image in a batch).

Basically we're reducing GPU RAM usage to nearly zero but we're greatly slowing down the process_batch function which just calls process_image for every image in the batch instead of processing the whole batch at Once.

Also, the process image is responible for normalizing and retyping from uint8 to float32. (the metadata dataset "images_md" is uint8)

In [ ]:
import torch

# this takes as input a batch of images_md of shape [32, 1+num_zones].
def process_image(metadata,noise, zones, overlaps, device="cuda"):
    """
    Process a single image from metadata and generate the corresponding RGB image.

    Args:
        metadata (torch.Tensor): Metadata for an image (shape [1 + num_zones])
        device (str): Device for computation (default is 'cuda')

    Returns:
        torch.Tensor: Processed image (shape [3, height, width])
    """
    metadata = metadata.to(device)  #set to cuda if not already set, this doesnt reduce efficiency by a lot if device is already cuda

    # Initialize output image (RGB)
    output_image = torch.zeros((3, sizey, sizex), device=device, dtype=torch.uint8)

    # set the seed for that specific image, smth like torch seed = metadata[0]
    torch.manual_seed(metadata[0].item())  # Unique seed for each image


    # Sum up activated zones
    for zone_id in range(len(zone_truthfulness_list)): # we must iterate over possible all zones !!! at least to generate the random number, even if we dont actually create the zone


        #generate the random colors for each zone even if were not adding it not to add an offset to the random number generator
        random_g = torch.randint(0, 256, (1,), dtype=torch.uint8, device=device).item() #generate one random value for the green channel for that zone
        random_b = torch.randint(0, 256, (1,), dtype=torch.uint8, device=device).item() #generate one random value for the blue channel for that zone

        if zone_id in zones:

            # Initialize a zone_layer tensor of shape (3, height, width)
            zone_layer = torch.zeros((3, sizey, sizex), dtype=torch.uint8, device=device)

            # then using the coordinates of the current zone's pixels: zone_coordinates = zone_pixels_list[zone_id]
            # Get the coordinates of the current zone's pixels
            zone_coordinates = zone_pixels_list[zone_id]
            zone_x, zone_y = zip(*zone_coordinates)


            zone_layer[1, zone_x, zone_y] = random_g
            zone_layer[2, zone_x, zone_y] = random_b

            #then we set all zone pixel's R channel value to metadata[1+zone_id].item()
            zone_layer[0, zone_x, zone_y] = metadata[1 + zone_id].item()

            # and finally add the layer to the output image like this:
            output_image += zone_layer

            #print(output_image.shape)
            #plt.imshow(output_image.to('cpu').permute(1,2,0))


    #output_image now contains the correct zones, need to add noise now

    #if there isnt noise just return the zone sum (which can also be empty)
    if noise == False:
        return output_image.float() / 255.0

    else: #case where noise is true
        #need to generate the noise layer
        noise_layer = torch.zeros((3, sizey, sizex), device=device, dtype=torch.uint8)
        noise_flat_indices = torch.randperm(sizex * sizey, device=device)[:n_noise_pixels]
        noise_x, noise_y = torch.div(noise_flat_indices, sizey, rounding_mode='floor'), noise_flat_indices % sizey
        noise_colors = torch.randint(0, 256, (n_noise_pixels, 3), dtype=torch.uint8, device=device)
        noise_layer[:, noise_x, noise_y] = noise_colors.permute(1, 0)
        #noise layer is a (3,h,w) rgb image


        if overlaps:  # If overlaps=True, take the noise as the base, and add the zones wherever the noise is zero
            isnt_noise_mask = (noise_layer == 0.0)  # get locations where noise is zero
            noise_layer[isnt_noise_mask] = output_image[isnt_noise_mask]  # Assign zones where noise is zero
            return noise_layer.float() / 255.0


        else: # If overlaps=False, take zones as base, then add noise only to pixels not covered by any zone
            isnt_zone_mask = (output_image == 0)  # Mask where any zone is present
            output_image[isnt_zone_mask] = noise_layer[isnt_zone_mask]  # Add noise only outside zones
            return output_image.float() / 255.0

## Process Batch function (just iterates over process image)

In [ ]:
def process_md_batch(metadata,noise, zones, overlaps, device="cuda"):
    returned_batch = torch.zeros((len(metadata), 3, sizey, sizex), device=device)
    for i, image_md in enumerate(metadata):
        returned_batch[i] = process_image(image_md, noise, zones, overlaps, device)
    return returned_batch

## Trainer function

In [ ]:
# Training function, add noise and zones to include as parameters
def train_model(model, train_loader, val_loader, num_epochs, zones_to_use, noise, overlaps,lr): #zones to use is a list of integers (maybe empty) and noise and overlaps are bool

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_loss, val_loss, train_acc, val_acc = [], [], [], []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss, running_corrects = 0.0, 0

        with tqdm(total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs} (Training)", unit="batch") as pbar:
          for input_batch, label_batch in train_loader:
              input_batch_processed = process_md_batch(input_batch, noise, zones_to_use, overlaps)
              #inputs, labels = inputs.to(device), labels.to(device)
              optimizer.zero_grad()
              outputs = model(input_batch_processed)
              loss = criterion(outputs, label_batch)
              loss.backward()
              optimizer.step()

              running_loss += loss.item() * input_batch_processed.size(0)
              _, preds = torch.max(outputs, 1)
              running_corrects += torch.sum(preds == label_batch)

              pbar.update(1)  # Update progress bar by one batch


        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)
        train_loss.append(epoch_loss)
        train_acc.append(epoch_acc.item())


        # Validation phase
        if val_loader is not None and len(val_loader) > 0:
            model.eval()
            val_running_loss, val_running_corrects = 0.0, 0
            with torch.no_grad():
                for input_batch, label_batch in val_loader:
                    input_batch_processed = process_md_batch(input_batch, noise, zones_to_use, overlaps)

                    #inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(input_batch_processed)
                    loss = criterion(outputs, label_batch)

                    val_running_loss += loss.item() * input_batch_processed.size(0)
                    _, preds = torch.max(outputs, 1)
                    val_running_corrects += torch.sum(preds == label_batch)

            val_epoch_loss = val_running_loss / len(val_loader.dataset)
            val_epoch_acc = val_running_corrects.double() / len(val_loader.dataset)
            val_loss.append(val_epoch_loss)
            val_acc.append(val_epoch_acc.item())



        if val_loader is not None and len(val_loader) > 0:
          print(f"Epoch {epoch+1}/{num_epochs}, "
                f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, "
                f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")
        else:
          print(f"Epoch {epoch+1}/{num_epochs}, "
                f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}")

    return train_loss, val_loss, train_acc, val_acc

## Evaluate function

In [ ]:
def evaluate_model(model, eval_loader, zones_to_use, noise, overlaps):
    model.eval()
    corrects = 0
    with torch.no_grad():
      for inputs, lab in tqdm(eval_loader, leave=False):
          inputs = process_md_batch(inputs, noise, zones_to_use, overlaps)
          inputs, cur_labels = inputs.to(device), lab.to(device)
          outputs = model(inputs)
          _, preds = torch.max(outputs, 1)
          corrects += torch.sum(preds == cur_labels)

    acc = corrects.double() / len(eval_loader.dataset)
    #print(f"Accuracy on {name}: {acc:.4f}")
    return acc.item()

## Run different evals function

In [ ]:
#evaluating on test set and on train set to check for overfitting
def run_evals(model):

    results = {}

    #eval on test and train
    results["test"] = evaluate_model(model, test_loader,zones_to_use=zones_for_train,noise=noise_train,overlaps=overlaps_train)
    results["train"] = evaluate_model(model, train_loader,zones_to_use=zones_for_train,noise=noise_train,overlaps=overlaps_train)
    print("\n")

    #removing noise
    results["test_no_noise"] = evaluate_model(model, test_loader,zones_to_use=zones_for_train,noise=False,overlaps=overlaps_train)
    print("\n")

    #influence of removing zones # subtractice importance
    for zone_to_remove in zones_for_train:
        results[f"test_without_zone_{zone_to_remove}"] = evaluate_model(model, test_loader,zones_to_use=[x for x in zones_for_train if x != zone_to_remove],noise=noise_train,overlaps=overlaps_train)

    #influence of keeping only single zones # additive importance
    for zone_to_use in zones_for_train:
        results[f"test_with_zone_{zone_to_use}"] = evaluate_model(model, test_loader,zones_to_use=[zone_to_use],noise=False,overlaps=overlaps_train)

    return results

Removing noise during inference disrupts the input space the model was trained on, causing degraded performance because the distribution of the data has changed.

We find that truthfulness makes is proportional to saliency !!! (according to subractive and additive viewpoints)

## Get average IoU of top N pixels and most truthful zone



---


For each sample:
Are the top N attributed pixels (if the most truthful zone is N pixels large) sufficient to correctly classify that image?
1.   NO: Is the zone as a whole sufficient to  correctly classify?

  * a. Yes:  Intersection over union. (there was a salient area that could've
  reached the correct prediction but the attribution pointed at something else)

  * b. No: Have a counter for how often this happens. Don't add towards IoU
    
2. YES: Have a counter for how often this happens. Count entire area as correct for the purpose of IoU


---


IoU is normalised by number of pixels in the zone and by number of samples that don't fall in case 1.b.

In [ ]:
def get_avg_iou(model):

    noise = noise_train
    zones=zones_for_train
    overlaps=overlaps_train

    saliency = Saliency(model)
    input_x_gradient = InputXGradient(model)
    integrated_gradients = IntegratedGradients(model)


    iou_lists = {'sal': [], 'ixg': [], 'ig': []}
    valid_iou_counters = {'sal': 0, 'ixg': 0, 'ig': 0}
    total_fail_counters = {'sal': 0, 'ixg': 0, 'ig': 0}
    correctly_attributed_counters= {'sal': 0, 'ixg': 0, 'ig': 0}



    most_truthful_zone_idx = np.argmax(zone_truthfulness_list)
    most_truthful_zone_pixels = set(zone_pixels_list[most_truthful_zone_idx])

    for test_batch in tqdm(test_loader, desc="Processing test loader"):
        test_images, test_labels = test_batch

        for i in range(len(test_images)):
            # test image shape is [B=1, 1 + num_zones, 224, 224, 3], removes batch when accessed then readded batch dim with unsqueeze
            test_image = test_images[i].unsqueeze(0) # unsqueeze to keep batch dimension # Shape: (B=1, 3, H, W).
            test_label = test_labels[i]

            test_image_processed = process_md_batch(test_image, noise, zones, overlaps,device=device)# Shape: (B=1, 3, H, W).
            width = test_image_processed.shape[-1]

            # define function that takes positions of top attrib pixels and edits counters
            def check_sample(top_attributed_pixels, attrib_method):#, test_image_processed = test_image_processed, most_truthful_zone_pixels=most_truthful_zone_pixels,model=model,test_label=test_label):
                assert isinstance(attrib_method, str) and attrib_method in {"sal", "ixg", "ig"}, "attrib_method must be a string and one of {'sal', 'ixg', 'ig'}"

                most_attr_pixels_only = test_image_processed.clone() # Shape: (B=1, 3, H, W)
                mask = torch.zeros_like(most_attr_pixels_only, dtype=torch.bool)
                for (x, y) in top_attributed_pixels:
                    x, y = x.item(), y.item()  # Convert tensor to Python integer
                    mask[:, :, x, y] = True
                most_attr_pixels_only[~mask] = 0 #pixels not in mask are set to 0 ie pixels that are not "both amongst the top N attributed and in zone"


                prediction = model(most_attr_pixels_only).argmax(dim=1).item()
                if prediction == test_label: ##CASE 2
                    correctly_attributed_counters[attrib_method] += 1
                    #print(f"Correctly Attributed for {attrib_method}: {correctly_attributed_counters[attrib_method]}")
                    iou_lists[attrib_method].append(1.0)

                else: ## CASE 1: most attr pixels arent enough

                    # create a mask that is true only on the pixels in the most truthful zone
                    whole_zone = test_image_processed.clone()
                    mask = torch.zeros_like(whole_zone, dtype=torch.bool)
                    for x, y in most_truthful_zone_pixels:
                        mask[:, :, x, y] = True
                    whole_zone[~mask] = 0 #pixels not in mask are set to 0 ie pixels not in most truthful zone

                    #run inference on whole_zone
                    zone_prediction = model(whole_zone).argmax(dim=1).item()
                    if zone_prediction == test_label: ## CASE 1.a. , we count the IoU
                        intersection = len(top_attributed_pixels.intersection(most_truthful_zone_pixels))
                        iou = intersection / len(most_truthful_zone_pixels) if most_truthful_zone_pixels else 0
                        iou_lists[attrib_method].append(iou)
                        valid_iou_counters[attrib_method] += 1

                    else:   ## CASE 1.b
                        total_fail_counters[attrib_method] += 1



            def RGB_attr_to_top_attr_pixels_pos(attr_RGB,width): #need width to turn flat pos into x,y pos
                attr = attr_RGB.max(dim=1, keepdim=True)[0] #shape: (b=1, c=1, h,w) #from 3 channels to 1 channel, now we can order by attr magnitude
                flat_top_attributed_indices = torch.argsort(attr.view(-1), descending=True)[:len(most_truthful_zone_pixels)]
                top_attributed_pixels = set(zip(flat_top_attributed_indices // width, flat_top_attributed_indices % width))
                return top_attributed_pixels





            # 1. Saliency
            attr_RGB_sal = saliency.attribute(test_image_processed, target=test_label).abs()#shape: (b=1, c=3, h,w)
            top_attributed_pixels_sal = RGB_attr_to_top_attr_pixels_pos(attr_RGB_sal,width)
            check_sample(top_attributed_pixels_sal, "sal")


            # 2. IxG
            attr_RGB_ixg = input_x_gradient.attribute(test_image_processed, target=test_label).abs()#shape: (b=1, c=3, h,w)
            top_attributed_pixels_ixg = RGB_attr_to_top_attr_pixels_pos(attr_RGB_ixg,width)
            check_sample(top_attributed_pixels_ixg, "ixg")


            # 3. IG
            attr_RGB_ig = integrated_gradients.attribute(test_image_processed, target=test_label, baselines=test_image_processed * 0).abs()#shape: (b=1, c=3, h,w)
            top_attributed_pixels_ig = RGB_attr_to_top_attr_pixels_pos(attr_RGB_ig,width)
            check_sample(top_attributed_pixels_ig, "ig")




    avg_ious = {}
    for attrib_method in iou_lists:
        avg_ious[attrib_method] = np.mean(iou_lists[attrib_method])

    results = {
        "sal": {
            "Avg Iou": avg_ious["sal"],
            "Correctly attributed": correctly_attributed_counters["sal"],
            "Total Fail counter": total_fail_counters["sal"],
            "Valid IoU counter": valid_iou_counters["sal"]
        },
        "ixg": {
            "Avg Iou": avg_ious["ixg"],
            "Correctly attributed": correctly_attributed_counters["ixg"],
            "Total Fail counter": total_fail_counters["ixg"],
            "Valid IoU counter": valid_iou_counters["ixg"]
        },
        "ig": {
            "Avg Iou": avg_ious["ig"],
            "Correctly attributed": correctly_attributed_counters["ig"],
            "Total Fail counter": total_fail_counters["ig"],
            "Valid IoU counter": valid_iou_counters["ig"]
        }
    }

    return results


#get_avg_iou(CNNmodel) #same thing here we use the default input values (same as train)


## Compare additive and subtractive importance of top N attributed pixels and most truthful zone

In [ ]:
def compare_importance(model):

    noise = noise_train
    zones=zones_for_train
    overlaps=overlaps_train

    saliency = Saliency(model)
    input_x_gradient = InputXGradient(model)
    integrated_gradients = IntegratedGradients(model)

    keep_zone_corrects = {'sal': 0, 'ixg': 0, 'ig': 0}
    keep_attr_pixels_corrects = {'sal': 0, 'ixg': 0, 'ig': 0}
    remove_zone_corrects = {'sal': 0, 'ixg': 0, 'ig': 0}
    remove_attr_pixels_corrects= {'sal': 0, 'ixg': 0, 'ig': 0}


    most_truthful_zone_idx = np.argmax(zone_truthfulness_list)
    most_truthful_zone_pixels = set(zone_pixels_list[most_truthful_zone_idx])

    for test_batch in tqdm(test_loader, desc="Processing test loader"):
        test_images, test_labels = test_batch

        for i in range(len(test_images)):
            # test image shape is [B=1, 1 + num_zones, 224, 224, 3], removes batch when accessed then readded batch dim with unsqueeze
            test_image = test_images[i].unsqueeze(0) # unsqueeze to keep batch dimension # Shape: (B=1, 3, H, W).
            test_label = test_labels[i]

            test_image_processed_full = process_md_batch(test_image, noise, zones, overlaps,device=device)# Shape: (B=1, 3, H, W). # the first infernece is ran with the same training settings (we dont add/remove anything)
            width = test_image_processed_full.shape[-1]






            def RGB_attr_to_top_attr_pixels_pos(attr_RGB,width): #need width to turn flat pos into x,y pos
                attr = attr_RGB.max(dim=1, keepdim=True)[0] #shape: (b=1, c=1, h,w) #from 3 channels to 1 channel, now we can order by attr magnitude
                flat_top_attributed_indices = torch.argsort(attr.view(-1), descending=True)[:len(most_truthful_zone_pixels)]
                top_attributed_pixels = set(zip(flat_top_attributed_indices // width, flat_top_attributed_indices % width))
                return top_attributed_pixels

            # define function that takes top attr pixels pos as input and edits counters as output
            def check_sample(image_md, full_processed_image, top_attributed_pixels,label, attrib_method):
                assert isinstance(attrib_method, str) and attrib_method in {"sal", "ixg", "ig"}, "attrib_method must be a string and one of {'sal', 'ixg', 'ig'}"


                attr_pixels_only_mask = torch.zeros_like(full_processed_image, dtype=torch.bool)

                ### first part: keep/remove the top attr pixels ###
                #init images
                keep_attr_pixels = full_processed_image.clone() # Shape: (B=1, 3, H, W) #init image with top_attr_pixels only
                remove_attr_pixels = full_processed_image.clone() # Shape: (B=1, 3, H, W) #init image with top_attr_pixels only


                # mask
                attr_pixels_only_mask = torch.zeros_like(full_processed_image, dtype=torch.bool)
                for (x, y) in top_attributed_pixels:
                    x, y = x.item(), y.item()  # Convert tensor to Python integer
                    attr_pixels_only_mask[:, :, x, y] = True
                remove_attr_pixels[attr_pixels_only_mask] = 0 #pixels where mask is true are set to 0 (ie remove top attr pixels)
                keep_attr_pixels[~attr_pixels_only_mask] = 0 #pixels where mask is false (~ operator) are set to 0


                remove_attr_pixels_prediction = model(remove_attr_pixels).argmax(dim=1).item()
                if remove_attr_pixels_prediction == label:
                    remove_attr_pixels_corrects[attrib_method] +=1

                keep_attr_pixels_prediction = model(keep_attr_pixels).argmax(dim=1).item()
                if keep_attr_pixels_prediction == label:
                    keep_attr_pixels_corrects[attrib_method] +=1



                ### second part: keep/remove most salient zone ###

                image_keep_zone = process_md_batch(image_md, False, [most_truthful_zone_idx], overlaps,device=device)
                image_remove_zone = process_md_batch(image_md, noise, [x for x in zones_for_train if x != most_truthful_zone_idx], overlaps,device=device)

                image_remove_zone_prediction = model(image_remove_zone).argmax(dim=1).item()
                if image_remove_zone_prediction == label:
                    remove_zone_corrects[attrib_method] +=1

                image_keep_zone_prediction = model(image_keep_zone).argmax(dim=1).item()
                if image_keep_zone_prediction == label:
                    keep_zone_corrects[attrib_method] +=1



            attr_RGB_sal = saliency.attribute(test_image_processed_full, target=test_label).abs()#shape: (b=1, c=3, h,w)
            top_attributed_pixels_sal = RGB_attr_to_top_attr_pixels_pos(attr_RGB_sal,width)
            check_sample(test_image, test_image_processed_full, top_attributed_pixels_sal,test_label, "sal")


            attr_RGB_ixg = input_x_gradient.attribute(test_image_processed_full, target=test_label).abs()#shape: (b=1, c=3, h,w)
            top_attributed_pixels_ixg = RGB_attr_to_top_attr_pixels_pos(attr_RGB_ixg,width)
            check_sample(test_image, test_image_processed_full, top_attributed_pixels_ixg,test_label, "ixg")


            attr_RGB_ig = integrated_gradients.attribute(test_image_processed_full, target=test_label, baselines=test_image_processed_full * 0).abs()
            top_attributed_pixels_ig = RGB_attr_to_top_attr_pixels_pos(attr_RGB_ig,width)
            check_sample(test_image, test_image_processed_full, top_attributed_pixels_ig,test_label, "ig")





    keep_zone_accuracy = {}
    keep_attr_pixels_accuracy = {}
    remove_zone_accuracy = {}
    remove_attr_pixels_accuracy= {}

    total_images = len(test_loader.dataset)

    attrib_methods = list(keep_zone_corrects.keys())

    for method in attrib_methods:
        keep_zone_accuracy[method] = keep_zone_corrects[method] / total_images * 100
        keep_attr_pixels_accuracy[method] = keep_attr_pixels_corrects[method] / total_images * 100
        remove_zone_accuracy[method] = remove_zone_corrects[method] / total_images * 100
        remove_attr_pixels_accuracy[method] = remove_attr_pixels_corrects[method] / total_images * 100

    # Instead of printing, create a structured JSON-like dictionary
    results = {}
    for method in attrib_methods:
        results[method] = {
            "KeepZone": keep_zone_accuracy[method],
            "Keep Attributed": keep_attr_pixels_accuracy[method],
            "Remove Zone": remove_zone_accuracy[method],
            "Remove Attributed": remove_attr_pixels_accuracy[method]
        }

    return results

This shows that there was at least one other subset of pixels that was more important than the attributed pixels. (here that subset is our most truthful zone). Note that we do not claim that that the zone is the most important. We're just showing that the attrib pixels aren't.

# Run experiment function



In [ ]:
def run_experiment(experiment_parameters,model_name):

    # Extract parameters from the dictionary

    ### Zones parameters ###
    global zone_truthfulness_list, zone_range_list, zone_pixels_list, all_zone_indices
    zone_truthfulness_list = experiment_parameters["zone_truthfulness_list"]
    zone_range_list = experiment_parameters["zone_range_list"]
    zone_pixels_list = experiment_parameters["zone_pixels_list"]
    all_zone_indices = experiment_parameters["all_zone_indices"]

    ### Dataset Variables ###
    global n_noise_pixels, n_images, sizex, sizey, n_classes
    n_noise_pixels = experiment_parameters["n_noise_pixels"]
    n_images = experiment_parameters["n_images"]
    sizex = experiment_parameters["sizex"]
    sizey = experiment_parameters["sizey"]
    n_classes = experiment_parameters["n_classes"]

    ### Training Variables ###
    global noise_train, overlaps_train, num_epochs, train_frac, val_frac, test_frac, batch_size, zones_for_train
    noise_train = experiment_parameters["noise_train"]
    overlaps_train = experiment_parameters["overlaps_train"]
    train_frac = experiment_parameters["train_frac"]
    val_frac = experiment_parameters["val_frac"]
    test_frac = experiment_parameters["test_frac"]
    batch_size = experiment_parameters["batch_size"]
    zones_for_train = experiment_parameters["zones_for_train"]

    assert len(zone_truthfulness_list) == len(zone_range_list) == len(zone_pixels_list)


    #STEP 1: Generate Images Metadata
    images_md,labels = generate_images_md(n_images, sizex, sizey, n_noise_pixels, zone_truthfulness_list,
                               zone_range_list, n_classes, device, 3) #the random seed, here 3, defines the randomized labels as well as the RNG on whether or not each zone is truthful, as well as the color in case of not truthful

    #STEP 2: Generate Dataset and then generate Loaders
    md_dataset = md_imageDataset(images_md, labels) #(num_images,1+numzone) #ie each image is represented by 1+ num zone int8 numbers
    global train_loader, val_loader, test_loader
    train_loader, val_loader, test_loader = create_data_loaders(md_dataset, train_frac, val_frac, test_frac, batch_size=batch_size,seed=3) #the loaders contains normalized float32 type images. # The seed is fort the partitioning when creating the loaders



    #STEP 3: Define and train  Model
    if model_name =="resnet":
      model = models.resnet50(weights=True)
      model.fc = nn.Linear(in_features=2048, out_features=n_classes)
      model = model.to(device)
      lr = 0.01
      num_epochs = 2
    elif model_name == "cnn":
      torch.manual_seed(4)
      model = SimpleCNN(n_classes,sizex,sizey).to(device)
      lr = 0.001
      num_epochs = 1
    elif model_name == "vgg":
      print("need to implement this one")
      return
    else :
      print("invalid model name")
      return

    print("training "+model_name)
    train_model(model, train_loader, val_loader, num_epochs=num_epochs,zones_to_use=zones_for_train,noise=noise_train,overlaps=overlaps_train,lr=lr)

    print("getting eval on test for "+model_name)
    #STEP 4: Run evals and check for additive and subtractive importance of zones for model
    eval_results = run_evals(model)
    #eval_results = evaluate_model(model, test_loader,zones_to_use=zones_for_train,noise=noise_train,overlaps=overlaps_train)

    print("getting avg iou for "+model_name)
    #STEP 5: Run attrib method evaluations using IoU method for model
    iou_results = get_avg_iou(model)

    print("getting importance for "+model_name)
    #STEP 6: Run attrib metod evaluation using Add and Sub importance for model
    importance_results = compare_importance(model)
    #importance_results = "poopoo"


    experiment_results = eval_results, iou_results, importance_results
    return experiment_results

# Run experiments function (to find best parameters)

In [ ]:
def run_experiments(parameters_list,csv_file_path,model_name):
    #headers for csv
    headers = [
        "zone_truthfulness_list", "zone_range_list", "other params",


        "test_eval", "train_eval", "test_no_noise_eval",
        "test_without_zone_0_eval", "test_without_zone_1_eval", "test_without_zone_2_eval",
        "test_with_zone_0_eval", "test_with_zone_1_eval", "test_with_zone_2_eval",

        "avg_iou_sal", "correctly_attributed_sal", "valid_iou_counter_sal","total_fail_sal",
        "avg_iou_ixg", "correctly_attributed_ixg", "valid_iou_counter_ixg","total_fail_ixg",
        "avg_iou_ig", "correctly_attributed_ig", "valid_iou_counter_ig","total_fail_ig",

        "keep_zone", "remove_zone",
        "keep_attributed_sal", "remove_attributed_sal",
        "keep_attributed_ixg", "remove_attributed_ixg",
        "keep_attributed_ig", "remove_attributed_ig",

        #"kzixg", "rzixg",
        #"kzig", "rzig",

    ]
    # Open the CSV file and write the headers
    file_exists = os.path.exists(csv_file_path)
    if not file_exists:
        with open(csv_file_path, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(headers)  # Write headers only once

    for parameters in tqdm(parameters_list):
        evals, iou_results, importance_results = run_experiment(parameters,model_name)

        # eval results are shaped:
        '''
        results = {
            "test": val,
            "train": val,
            "test_no_noise": val,
            "test_without_zone_{zone_removed}": val,
            ....
            "test_with_zone_{zone_kept}": val,
            ....
        }
        '''

        #iou_results_CNN and resnet are formatted as follows:
        '''
        results = {
            "sal": {
                "Avg Iou": value,
                "Correctly attributed": value,
                "Total Fail counter": value,
                "Valid IoU counter": value
            },
            "ixg": {
                "Avg Iou": value,
                "Correctly attributed": value,
                "Total Fail counter": value,
                "Valid IoU counter": value
            },
            "ig": {
                "Avg Iou": value,
                "Correctly attributed": value,
                "Total Fail counter": value,
                "Valid IoU counter": value
            }
        }
        '''
        # and importance results for cnn and resnet are formatted like this:
        '''
        results = {
            "sal": {
                "KeepZone": val,
                "Keep Attributed": val,
                "Remove Zone": val,
                "Remove Attributed": val
            },
            "ixg": {
                "KeepZone": val,
                "Keep Attributed": val,
                "Remove Zone": val,
                "Remove Attributed": val
            },
            "ig": {
                "KeepZone": avg_iousval,
                "Keep Attributed": val,
                "Remove Zone": val,
                "Remove Attributed": val
            }
        }
        '''
        ##append csv. each experiment fits on a single line.
        other_parameters = {k: v for k, v in parameters.items() if k not in ["zone_truthfulness_list", "zone_range_list"]}
        row = [
            parameters["zone_truthfulness_list"],  # First column
            parameters["zone_range_list"],  # Second column
            #add a dictionnary with all other parameters (will fit in a single column)
            json.dumps(other_parameters),  # Third column


            ## EVAL RESULTS
            #test_eval,
            evals["test"], evals['train'], evals['test_no_noise'],
            evals['test_without_zone_0'], evals['test_without_zone_1'], evals['test_without_zone_2'],
            evals['test_with_zone_0'], evals['test_with_zone_1'], evals['test_with_zone_2'],



            # IoU RESULTS
            iou_results["sal"]["Avg Iou"],
            iou_results["sal"]["Correctly attributed"],
            iou_results["sal"]["Valid IoU counter"],
            iou_results["sal"]["Total Fail counter"],
            iou_results["ixg"]["Avg Iou"],
            iou_results["ixg"]["Correctly attributed"],
            iou_results["ixg"]["Valid IoU counter"],
            iou_results["ixg"]["Total Fail counter"],
            iou_results["ig"]["Avg Iou"],
            iou_results["ig"]["Correctly attributed"],
            iou_results["ig"]["Valid IoU counter"],
            iou_results["ig"]["Total Fail counter"],

            # Importance results
            importance_results["sal"]["KeepZone"],
            importance_results["sal"]["Remove Zone"],

            importance_results["sal"]["Keep Attributed"],
            importance_results["sal"]["Remove Attributed"],
            importance_results["ixg"]["Keep Attributed"],
            importance_results["ixg"]["Remove Attributed"],
            importance_results["ig"]["Keep Attributed"],
            importance_results["ig"]["Remove Attributed"],

            #importance_results["ixg"]["KeepZone"],
            #importance_results["ixg"]["Remove Zone"],
            #importance_results["ig"]["KeepZone"],
            #importance_results["ig"]["Remove Zone"],

        ]

        # Open the CSV file in append mode and write the row
        with open(csv_file_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(row)

# Parameter configurations

we will only try to change the truthfulnesses and the ranges.

In [ ]:
experiment_parameters_template= {
    ### Dataset Zone variables ###
    "zone_pixels_list" : [
        # Zone 1: Centered at (60, 60), 10x10 square
        [(x, y) for x in range(55, 65) for y in range(55, 65)],
        # Zone 2: Centered at (180, 180), 10x10 square
        [(x, y) for x in range(175, 185) for y in range(175, 185)],
        [(x, y) for x in range(90, 100) for y in range(120, 130)],
        #[(x, y) for x in range(170, 180) for y in range(20, 30)],
        #[(x, y) for x in range(10, 20) for y in range(180, 190)],
    ],
    "zone_truthfulness_list" : [0.5, 0.93, 0.5],
    "zone_range_list" :  [(10, 90), (10, 140), (10, 90)],  # Define range for each zone
    "all_zone_indices" : [0,1,2],


    ### Dataset Variables ###
    "n_noise_pixels" : 6000, #set to 0 if never used
    "n_images" : 15000, #10k
    "sizex" : 224,
    "sizey" : 224,  #for vgg or resnet
    "n_classes" : 5,


    ### Training variables ###
    "noise_train" : True, #True #noise is used for training
    "overlaps_train" : False, #noise overlaps over zones for training is what is used for training
    "train_frac" : 0.9,
    "val_frac" : 0,
    "test_frac" : 0.1,
    "batch_size" : 32,
    "zones_for_train" : [0,1,2],#,2]
}

## Create random experiments list to try

In [ ]:
zt = [[[x,x+delta,x] for delta in range(1,100-x,3)] for x in range(20,90,5)] #truthfulnesses
#zt = [[[x,x+delta,x] for delta in range(40,46,1)] for x in range(45,55,1)] #truthfulnesses
zr = [[[(10,x),(10,x+delta),(10,x)] for delta in range(20,250-x,5)] for x in range(60,150,10)] #steps, defined by the upper range
#zr = [[[(10,x),(10,x+delta),(10,x)] for delta in range(40,60,1)] for x in range(80,100,1)] #steps, defined by the upper range

flat_zt = list(itertools.chain.from_iterable(zt))
flat_zr = list(itertools.chain.from_iterable(zr))

# Divide all numbers by 100 or get zt usable percentages
normalized_zt = [[num / 100 for num in triplet] for triplet in flat_zt]
print(len(normalized_zt))
print(len(flat_zr))


print(normalized_zt)
print(flat_zr)

def generate_random_experiment_parameters(N, normalized_zt, flat_zr, seed=42):
    random.seed(seed)
    experiment_list = []
    for _ in range(N):
        exp_params = copy.deepcopy(experiment_parameters_template)
        exp_params["zone_truthfulness_list"] = random.choice(normalized_zt)
        exp_params["zone_range_list"] = random.choice(flat_zr)
        experiment_list.append(exp_params)
    return experiment_list


experiment_list = generate_random_experiment_parameters(500, normalized_zt, flat_zr)
#print(experiment_list[9]["zone_truthfulness_list"])
#print(experiment_list[9]["zone_range_list"])

222
234
[[0.2, 0.21, 0.2], [0.2, 0.24, 0.2], [0.2, 0.27, 0.2], [0.2, 0.3, 0.2], [0.2, 0.33, 0.2], [0.2, 0.36, 0.2], [0.2, 0.39, 0.2], [0.2, 0.42, 0.2], [0.2, 0.45, 0.2], [0.2, 0.48, 0.2], [0.2, 0.51, 0.2], [0.2, 0.54, 0.2], [0.2, 0.57, 0.2], [0.2, 0.6, 0.2], [0.2, 0.63, 0.2], [0.2, 0.66, 0.2], [0.2, 0.69, 0.2], [0.2, 0.72, 0.2], [0.2, 0.75, 0.2], [0.2, 0.78, 0.2], [0.2, 0.81, 0.2], [0.2, 0.84, 0.2], [0.2, 0.87, 0.2], [0.2, 0.9, 0.2], [0.2, 0.93, 0.2], [0.2, 0.96, 0.2], [0.2, 0.99, 0.2], [0.25, 0.26, 0.25], [0.25, 0.29, 0.25], [0.25, 0.32, 0.25], [0.25, 0.35, 0.25], [0.25, 0.38, 0.25], [0.25, 0.41, 0.25], [0.25, 0.44, 0.25], [0.25, 0.47, 0.25], [0.25, 0.5, 0.25], [0.25, 0.53, 0.25], [0.25, 0.56, 0.25], [0.25, 0.59, 0.25], [0.25, 0.62, 0.25], [0.25, 0.65, 0.25], [0.25, 0.68, 0.25], [0.25, 0.71, 0.25], [0.25, 0.74, 0.25], [0.25, 0.77, 0.25], [0.25, 0.8, 0.25], [0.25, 0.83, 0.25], [0.25, 0.86, 0.25], [0.25, 0.89, 0.25], [0.25, 0.92, 0.25], [0.25, 0.95, 0.25], [0.25, 0.98, 0.25], [0.3, 0.31

# Run experiments.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
### Check for GPU ###
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

parameters_list = experiment_list[300:]
#parameters_list = [experiment_parameters_template]


csv_file_path = "/content/drive/MyDrive/pfe_louis/resnet_all_attr.csv"


run_experiments(parameters_list, csv_file_path,"resnet")

Using device: cuda


  0%|          | 0/200 [00:00<?, ?it/s]

Generating zone metadata: 0it [00:00, ?it/s]

Generated metadata for 15000 images with 3 zones each.


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


...

getting importance for resnet


Processing test loader:   0%|          | 0/47 [00:00<?, ?it/s]

Generating zone metadata: 0it [00:00, ?it/s]

Generated metadata for 15000 images with 3 zones each.
training resnet


Epoch 1/2 (Training):   0%|          | 0/422 [00:00<?, ?batch/s]